# Extract Reddit Features

This notebook adds two features to Reddit data:
- **word_count**: Number of words in each post/comment
- **author_created_utc**: Account creation date of the post author

In [ ]:
# Import required libraries
import pandas as pd
import praw
import glob
import time
from datetime import datetime
from prawcore import TooManyRequests

## Configuration

Initialize Reddit API client.

In [ ]:
# Initialize Reddit API client
reddit = praw.Reddit(
    user_agent=True,
    client_id="",
    client_secret="",
    username="",
    password=""
)
print("Reddit client initialized")

## Helper Functions

Functions for timestamp conversion, word counting, and fetching author data.

In [ ]:
def unix_to_datetime(unix_time):
    """Convert Unix timestamp to datetime string."""
    return datetime.fromtimestamp(unix_time).strftime('%Y-%m-%d %H:%M:%S')

def get_word_count(text):
    """Calculate word count from text (post or comment)."""
    if pd.isnull(text) or text == '[deleted]':
        return 0
    return len([word for word in str(text).split() if word])

def get_author_creation_date(author, reddit, cache):
    """Fetch author's account creation date, with caching and rate limit handling."""
    if author == '[deleted]' or pd.isnull(author):
        return ''
    
    # Check cache
    if author in cache:
        return cache[author]
    
    retry_count = 0
    max_retries = 3
    while retry_count < max_retries:
        try:
            redditor = reddit.redditor(author)
            creation_date = redditor.created_utc if redditor.created_utc else ''
            if creation_date:
                creation_date = unix_to_datetime(creation_date)
            cache[author] = creation_date
            time.sleep(1)  # Respect rate limits
            return creation_date
        except TooManyRequests:
            retry_count += 1
            print(f"Rate limit hit for author {author} (retry {retry_count}/{max_retries}). Waiting 120 seconds...")
            time.sleep(120)
            continue
        except Exception as e:
            print(f"Error fetching author {author}: {e}")
            cache[author] = ''
            return ''
    return ''

## Process Data

Load CSV, add word_count and author_created_utc columns, then save.

In [13]:
# Specify the CSV file to process
csv_file = 'GTLB/GTLB2.csv'  # Replace with your CSV file name

# Cache for author creation dates
author_cache = {}

print(f"Processing {csv_file}...")
try:
    # Load CSV
    df = pd.read_csv(csv_file)
    
    # Add word_count (from text only)
    df['word_count'] = df['text'].apply(get_word_count)
    
    # Add author_created_utc (line by line, with caching)
    df['author_created_utc'] = df['author'].apply(
        lambda x: get_author_creation_date(x, reddit, author_cache)
    )
    
    # Ensure all expected columns
    expected_columns = [
        'type', 'id', 'parent_post_id', 'subreddit', 'title', 'text',
        'url', 'upvotes', 'created_utc', 'author','author_link_karma', 'author_comment_karma',  'author_created_utc','word_count'
    ]
    for col in expected_columns:
        if col not in df.columns:
            df[col] = ''
    df = df[expected_columns]
    
    # Save updated CSV
    output_file = csv_file#.replace('S&P 100/', 'S&P Updated/').replace('.csv', '_updated.csv')
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Saved updated file: {output_file}")
    print(f"Rows: {len(df)}, Unique authors: {len(df['author'].unique())}")
    print(f"Author cache size: {len(author_cache)}")
    
except Exception as e:
    print(f"Error processing {csv_file}: {e}")

Processing GTLB/GTLB2.csv...
Saved updated file: GTLB/GTLB2.csv
Rows: 212, Unique authors: 162
Author cache size: 162


In [ ]:
# Process additional files if needed

# Uncomment below to process multiple files:#         # (same processing code as above)

#         print(f"\nProcessing {csv_file}...")

# import os#     if os.path.exists(csv_file):

# csv_files = ['GTLB/GTLB2.csv', 'ETH.csv']  # Add more files as needed# for csv_file in csv_files: